# Modul 11: Deteksi Anomali Smart Meter (Isolation Forest & LSTM Autoencoder)
**Mata Kuliah:** Analisis Deret Waktu dan Peramalan (TEE-412) — S1 Teknik Elektro FT UNIB  
**Sub-CPMK 12 (Minggu 13):** Merancang detektor anomali tanpa pengawasan (*unsupervised*) berbasis galat rekonstruksi Autoencoder ($\tau = \mu + 3\sigma$) dan mengevaluasinya menggunakan Precision, Recall, F1-Score, dan AUROC pada data *class imbalance*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import torch
import torch.nn as nn

torch.manual_seed(42)
np.random.seed(42)

## 1. Simulasi Profil Konsumsi Harian 24-Jam Smart Meter (Normal vs. Pencurian Listrik)

In [ ]:
hours = np.arange(24)
base_profile = 3.5 + 1.8 * np.sin(2 * np.pi * (hours - 7) / 24) + 1.5 * np.exp(-((hours - 19) ** 2) / 6)

# Data latih HANYA berisi profil normal (unsupervised)
n_train = 400
X_train = np.array([base_profile + np.random.normal(0, 0.25, 24) for _ in range(n_train)], dtype=np.float32)

# Data uji: 180 normal + 20 anomali pencurian/bypass (10% imbalance)
X_test_norm = np.array([base_profile + np.random.normal(0, 0.25, 24) for _ in range(180)], dtype=np.float32)
X_test_anom = np.array([base_profile * 0.35 + np.random.normal(0, 0.15, 24) for _ in range(20)], dtype=np.float32)
X_test = np.vstack([X_test_norm, X_test_anom])
y_true = np.array([0] * 180 + [1] * 20)  # 0=Normal, 1=Anomali

## 2. Pelatihan LSTM Autoencoder & Penentuan Ambang $\tau = \mu + 3\sigma$

In [ ]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.encoder = nn.LSTM(1, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, 1, batch_first=True)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        repeated = h.repeat(x.size(1), 1, 1).permute(1, 0, 2)
        recon, _ = self.decoder(repeated)
        return recon

ae = LSTMAutoencoder(hidden_dim=16)
opt = torch.optim.Adam(ae.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

X_tr_t = torch.from_numpy(X_train).unsqueeze(-1)
X_te_t = torch.from_numpy(X_test).unsqueeze(-1)

for epoch in range(40):
    opt.zero_grad()
    loss = loss_fn(ae(X_tr_t), X_tr_t)
    loss.backward()
    opt.step()

ae.eval()
with torch.no_grad():
    tr_recon = ae(X_tr_t).numpy().squeeze(-1)
    te_recon = ae(X_te_t).numpy().squeeze(-1)

tr_mse = np.mean((X_train - tr_recon) ** 2, axis=1)
te_mse = np.mean((X_test - te_recon) ** 2, axis=1)

threshold = tr_mse.mean() + 3.0 * tr_mse.std()
y_pred_ae = (te_mse > threshold).astype(int)

# Bandingkan dengan Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42).fit(X_train)
y_pred_iso = (iso.predict(X_test) == -1).astype(int)

for name, yp, score in [('LSTM Autoencoder', y_pred_ae, te_mse), ('Isolation Forest', y_pred_iso, -iso.score_samples(X_test))]:
    print(f"=== {name} ===")
    print(f"  Precision={precision_score(y_true, yp):.3f} | Recall={recall_score(y_true, yp):.3f} | F1={f1_score(y_true, yp):.3f} | AUROC={roc_auc_score(y_true, score):.3f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(y_true, yp)}")

plt.figure(figsize=(10, 4))
plt.hist(te_mse[y_true == 0], bins=25, alpha=0.7, label='Normal', color='steelblue')
plt.hist(te_mse[y_true == 1], bins=15, alpha=0.7, label='Anomali Pencurian', color='crimson')
plt.axvline(threshold, color='black', ls='--', lw=2, label=f'Ambang tau = {threshold:.3f}')
plt.title('Distribusi Galat Rekonstruksi LSTM Autoencoder')
plt.xlabel('MSE Rekonstruksi')
plt.legend()
plt.grid(True)
plt.show()

## 3. Tugas & Pertanyaan Analisis (C4–C6)
1. **[C4]** Bandingkan kurva rekonstruksi sampel normal vs. sampel pencurian listrik. Mengapa galat rekonstruksi melonjak pada sampel pencurian?
2. **[C5]** Lakukan sapuan ambang $k \in \{1.5, 2.0, 2.5, 3.0, 3.5\}$ pada $\tau = \mu + k\sigma$ dan plot *trade-off* Precision vs. Recall!